# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities, including record sets and fields, are referenced by their Croissant `@id`, ensuring consistency and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and inspect basic dataset information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary metadata
metadata = dataset.metadata
print("Dataset: {}".format(getattr(metadata, 'name', metadata.__dict__.get('name', 'N/A'))))
print("Description: {}".format(getattr(metadata, 'description', metadata.__dict__.get('description', 'N/A'))))

## 2. Data Overview
Let's enumerate available record sets, their fields, and corresponding Croissant `@id` values so you can select entities for data extraction.

We use mlcroissant’s metadata interface to list all record sets and, for each, their fields and data types.

In [ ]:
# Helper: pretty-print for metadata
from mlcroissant.types import Field
def print_record_sets(md):
    if hasattr(md, 'record_sets'):
        for rs in md.record_sets:
            print(f"RecordSet Name: {getattr(rs, 'name', '')}")
            print(f"  @id: {getattr(rs, '@id', '')}")
            print(f"  Description: {getattr(rs, 'description', '')}")
            if hasattr(rs, 'fields'):
                print("  Fields:")
                for fld in rs.fields:
                    print(f"    - Name: {getattr(fld, 'name', '')}\n      @id: {getattr(fld, '@id', '')}\n      DataType: {getattr(fld, 'data_type', '')}")
    else:
        print("No record sets found in metadata.")

print_record_sets(metadata)

## 3. Data Extraction
Load data from any available record set into a DataFrame using its `@id`. For the FAIR^2 dataset, we show how to extract all records for a chosen record set. Modify `record_sets_ids` to include only the desired `@id` of the record sets you wish to extract.

**Tip**: This example uses the (likely) primary record set, usually with protein summary or abundance data. Check the printed record set list above and substitute the value for your target.

In [ ]:
# Collect all record set @ids
record_sets_ids = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_sets_ids.append(getattr(rs, '@id', None))
print(f"Available RecordSet @ids: {record_sets_ids}")

# Will use the first record set as a main example (edit as needed!):
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
else:
    raise ValueError("No record sets found in this dataset.")

In [ ]:
# Load records from each record set into a DataFrame
dataframes = {}
for rec_id in record_sets_ids:
    print(f"Loading records for: {rec_id}")
    # The mlcroissant generator yields dicts for each row
    rows = list(dataset.records(record_set=rec_id))
    if rows:
        df = pd.DataFrame(rows)
        dataframes[rec_id] = df
        print(f"  Number of records: {len(df)}")
        print(f"  Columns (@id): {list(df.columns)}")
        display(df.head())
    else:
        print(f"  No records found for {rec_id}")

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate some simple numerical processing on this record set.

Replace the `numeric_field_id` and `group_field_id` below by choosing from columns above (which are always Croissant `@id`s).

In [ ]:
# Example: select a numeric field and group field from the dataframe
# Replace these with relevant field @id strings seen above
example_df = dataframes[main_record_set_id]
numeric_field_id = None
group_field_id = None
# Try to auto-select a numeric column
for col in example_df.columns:
    if example_df[col].dtype in ['float64', 'int64']:
        numeric_field_id = col
        break
if not numeric_field_id:
    # If nothing typed as numeric, try to coerce one
    for col in example_df.columns:
        try:
            example_df[col] = pd.to_numeric(example_df[col])
            numeric_field_id = col
            break
        except:
            continue
if not numeric_field_id:
    raise ValueError("No numeric field could be identified in the chosen record set.")

# Try to select a group-able categorical field (e.g. protein name, or other string field)
for col in example_df.columns:
    if col != numeric_field_id and example_df[col].dtype == 'object':
        group_field_id = col
        break
print(f"Selected numeric field (@id): {numeric_field_id}")
print(f"Selected group/categorical field (@id): {group_field_id}")

# EDA: Filter, normalize, group
threshold = example_df[numeric_field_id].mean() if numeric_field_id else 0
filtered_df = example_df[example_df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the selected field
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(f"Grouped by {group_field_id}, mean {numeric_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and explore groupwise means.

*You can further customize the fields visualized using their Croissant `@id` as above.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(example_df[numeric_field_id].dropna(), bins=30)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# If grouped data exists, plot top 10 mean values
if 'grouped_df' in locals() and grouped_df is not None:
    plt.figure(figsize=(8,4))
    grouped_df = grouped_df.sort_values(by=f'mean_{numeric_field_id}', ascending=False).head(10)
    sns.barplot(y=grouped_df.index, x=grouped_df[f'mean_{numeric_field_id}'])
    plt.title(f'Top 10 Groups by Mean {numeric_field_id}')
    plt.xlabel(f'Mean {numeric_field_id}')
    plt.ylabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to access, explore, and analyze the FAIR^2 mass spectrometry dataset using the Croissant `@id` structure and `mlcroissant` APIs. You can now customize the workflow to focus on downstream biological or analytical questions, always referencing fields and record sets by their unique IDs to support reproducibility and transparent data provenance.